In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 19:01:46 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 192.168.13.159 instead (on interface en0)
26/04/23 19:01:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 19:01:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import os
import requests

def download_file(url: str, output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

In [3]:
import gzip
import shutil

def gunzip_file(input_path: str, output_path: str):
    with gzip.open(input_path, "rb") as f_in:
        with open(output_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

In [4]:
years = [2023, 2024, 2025, 2026]

base_url = "https://nvd.nist.gov/feeds/json/cve/2.0"

raw_dir = "../data/raw/nvd"
json_dir = "../data/raw/nvd_json"

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(json_dir, exist_ok=True)

for year in years:
    gz_url = f"{base_url}/nvdcve-2.0-{year}.json.gz"
    gz_path = f"{raw_dir}/nvdcve-2.0-{year}.json.gz"
    json_path = f"{json_dir}/nvdcve-2.0-{year}.json"

    print(f"Downloading {year}...")
    download_file(gz_url, gz_path)

    print(f"Extracting {year}...")
    gunzip_file(gz_path, json_path)

Extracting 2023...
Extracting 2024...
Extracting 2025...
Extracting 2026...


In [2]:
nvd_raw = (
    spark.read
    .option("multiLine", "true")
    .json("../data/raw/nvd_json/*.json")
)

nvd_raw.printSchema()

26/04/23 19:01:58 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/nvd_json/*.json.
java.io.FileNotFoundException: File ../data/raw/nvd_json/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.

root
 |-- format: string (nullable = true)
 |-- resultsPerPage: long (nullable = true)
 |-- startIndex: long (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- totalResults: long (nullable = true)
 |-- version: string (nullable = true)
 |-- vulnerabilities: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- cve: struct (nullable = true)
 |    |    |    |-- cisaActionDue: string (nullable = true)
 |    |    |    |-- cisaExploitAdd: string (nullable = true)
 |    |    |    |-- cisaRequiredAction: string (nullable = true)
 |    |    |    |-- cisaVulnerabilityName: string (nullable = true)
 |    |    |    |-- configurations: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- nodes: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- cpeMatch: array (nullable = true)
 |    |    |    |    |    |

In [3]:
from pyspark.sql.functions import explode, col

nvd_exploded = (
    nvd_raw
    .select(explode(col("vulnerabilities")).alias("v"))
    .select("v.cve.*")
)

nvd_exploded.printSchema()

root
 |-- cisaActionDue: string (nullable = true)
 |-- cisaExploitAdd: string (nullable = true)
 |-- cisaRequiredAction: string (nullable = true)
 |-- cisaVulnerabilityName: string (nullable = true)
 |-- configurations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- nodes: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- cpeMatch: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- criteria: string (nullable = true)
 |    |    |    |    |    |    |-- matchCriteriaId: string (nullable = true)
 |    |    |    |    |    |    |-- versionEndExcluding: string (nullable = true)
 |    |    |    |    |    |    |-- versionEndIncluding: string (nullable = true)
 |    |    |    |    |    |    |-- versionStartExcluding: string (nullable = true)
 |    |    |    |    |    |    |-- versionStartIncluding: string (nullable = true)
 |

In [6]:
nvd_exploded.select("id").show(5)

+--------------+
|            id|
+--------------+
| CVE-2024-0069|
| CVE-2024-0070|
|CVE-2024-21732|
| CVE-2024-0181|
| CVE-2024-0182|
+--------------+
only showing top 5 rows


In [4]:
from pyspark.sql.functions import expr

nvd_df = (
    nvd_exploded
    .select(
        col("id").alias("cve_id"),
        col("published"),
        col("lastModified"),
        expr("descriptions[0].value").alias("description"),
        expr("weaknesses[0].description[0].value").alias("cwe"),
        expr("metrics.cvssMetricV31[0].cvssData.baseScore").alias("cvss_score"),
        expr("metrics.cvssMetricV31[0].cvssData.baseSeverity").alias("cvss_severity")
    )
)

nvd_df.show(5, truncate=False)

+--------------+-----------------------+-----------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+----------+-------------+
|cve_id        |published              |lastModified           |description                                                                                                                                                                                                                                                                                                                                                      

In [5]:
nvd_df.write.mode("overwrite").parquet("../data/silver/nvd")